# Analisis Rantai Pasok & Peramalan Permintaan E-Commerce Indonesia (Data Riil)

Notebook ini memproses data penjualan riil dari **UCI Online Retail Dataset** (541.909 transaksi) yang disaring khusus untuk produk terlaris (`StockCode 85123A`, dipetakan sebagai **`SKU_HIJAB_PREMIUM`**). 

Data ditransformasikan ke konteks pasar Indonesia dengan:
1. Konversi harga ke **Rupiah (IDR)** dengan kurs Rp20.000 / GBP.
2. Pergeseran tahun ke **2024-2025** agar relevan untuk analisis saat ini.
3. Simulasi kendala logistik lokal (**Stockout**) akibat banjir musiman dan antrean kurir.

Visualisasi hasil olahan diekspor langsung ke folder `images/` dengan resolusi 300 DPI untuk laporan bisnis utama di `README.md`.


In [1]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Konfigurasi visualisasi profesional
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16,
    'figure.figsize': (12, 6)
})

# Path data & images
DATA_DIR = "./data"
IMAGES_DIR = "./images"
print("Pustaka dan parameter visualisasi siap.")

Pustaka dan parameter visualisasi siap.


In [2]:
# Muat dataset mentah hasil ekstraksi transaksi riil
df_raw = pd.read_csv(os.path.join(DATA_DIR, "raw_demand.csv"))
df_raw['Date'] = pd.to_datetime(df_raw['Date'])
print(f"Data riil berhasil dimuat. Rentang tanggal: {df_raw['Date'].min().strftime('%Y-%m-%d')} s/d {df_raw['Date'].max().strftime('%Y-%m-%d')}")
df_raw.head()

Data riil berhasil dimuat. Rentang tanggal: 2024-12-01 s/d 2025-12-09


,Date,Product,Category,Price_IDR,Promotion,Demand_Raw,Stockout_Occurred,Sales_Actual
0,2024-12-01,SKU_HIJAB_PREMIUM,Fashion,61729,0,454,0,454
1,2024-12-02,SKU_HIJAB_PREMIUM,Fashion,53526,1,309,0,309
2,2024-12-03,SKU_HIJAB_PREMIUM,Fashion,67457,0,25,0,25
3,2024-12-04,SKU_HIJAB_PREMIUM,Fashion,67457,0,0,0,0
4,2024-12-05,SKU_HIJAB_PREMIUM,Fashion,57769,1,198,0,198


### Prapemrosesan Data (Imputasi Stockout)

Gangguan pengiriman menyebabkan penjualan aktual tercatat 0 di beberapa hari. Kita mendeteksi hari stockout ini dan mengimputasinya menggunakan **rata-rata bergerak 7 hari sebelumnya (*7-day rolling mean*)** untuk memulihkan permintaan pasar riil (*unconstrained demand*).


In [3]:
df_clean = df_raw.copy()

# Buat kolom target permintaan bersih
df_clean['Demand_Cleaned'] = df_clean['Sales_Actual']

# Hitung rata-rata bergerak 7 hari sebelumnya
rolling_impute = df_clean['Sales_Actual'].rolling(window=7, min_periods=1, closed='left').mean()

# Ganti nilai penjualan 0 pada hari stockout dengan nilai rata-rata bergerak
df_clean.loc[df_clean['Stockout_Occurred'] == 1, 'Demand_Cleaned'] = rolling_impute

# Dibulatkan ke integer dan hilangkan NaN jika ada di baris-baris pertama
df_clean['Demand_Cleaned'] = df_clean['Demand_Cleaned'].fillna(method='bfill').round().astype(int)

# Simpan data bersih
cleaned_file_path = os.path.join(DATA_DIR, "cleaned_demand.csv")
df_clean.to_csv(cleaned_file_path, index=False)
print("Data bersih berhasil disimpan.")
df_clean[df_clean['Stockout_Occurred'] == 1].head()

Data bersih berhasil disimpan.


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_15396\2243015422.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[171.57142857   0.          89.85714286  89.42857143 361.57142857
 333.28571429  73.28571429 171.57142857  70.71428571  60.14285714
  50.71428571  64.28571429  23.57142857 118.71428571]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_clean.loc[df_clean['Stockout_Occurred'] == 1, 'Demand_Cleaned'] = rolling_impute
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_15396\2243015422.py:13: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_clean['Demand_Cleaned'] = df_clean['Demand_Cleaned'].fillna(method='bfill').round().astype(int)


,Date,Product,Category,Price_IDR,Promotion,Demand_Raw,Stockout_Occurred,Sales_Actual,Demand_Cleaned
11,2024-12-12,SKU_HIJAB_PREMIUM,Fashion,59000,1,42,1,0,172
34,2025-01-04,SKU_HIJAB_PREMIUM,Fashion,66314,0,67,1,0,0
50,2025-01-20,SKU_HIJAB_PREMIUM,Fashion,59000,0,42,1,0,90
69,2025-02-08,SKU_HIJAB_PREMIUM,Fashion,64829,0,153,1,0,89
139,2025-04-19,SKU_HIJAB_PREMIUM,Fashion,63369,0,110,1,0,362


### Eksplorasi Data (EDA) & Dekomposisi Musiman

Kita menganalisis tren harian dan menguraikannya menjadi komponen tren, musiman mingguan (period=7), dan residu.


In [4]:
# 1. Plot Tren Permintaan Historis Riil
plt.figure(figsize=(12, 6))
plt.plot(df_clean['Date'], df_clean['Demand_Cleaned'], label='Permintaan Bersih (Imputasi)', color='#2980b9', alpha=0.85, linewidth=1.5)
plt.plot(df_clean['Date'], df_clean['Sales_Actual'], label='Penjualan Aktual Tercatat (Stockout)', color='#e67e22', alpha=0.4, linestyle=':')
plt.title('Tren Permintaan SKU_HIJAB_PREMIUM Agregat Nasional (Data Riil UCI)', fontsize=14, fontweight='bold')
plt.xlabel('Tanggal', fontsize=12)
plt.ylabel('Volume Unit', fontsize=12)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "demand_trend.png"), dpi=300)
plt.close()

# 2. Dekomposisi Musiman Mingguan (Weekly)
df_ts = df_clean.set_index('Date')
decomposition = seasonal_decompose(df_ts['Demand_Cleaned'], model='additive', period=7)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
decomposition.observed.plot(ax=axes[0], color='#2c3e50', legend=False)
axes[0].set_ylabel('Observed')
axes[0].set_title('Dekomposisi Aditif Permintaan Riil SKU_HIJAB_PREMIUM', fontsize=14, fontweight='bold')

decomposition.trend.plot(ax=axes[1], color='#c0392b', legend=False)
axes[1].set_ylabel('Trend')

decomposition.seasonal.plot(ax=axes[2], color='#8e44ad', legend=False)
axes[2].set_ylabel('Seasonal (7-Day)')

decomposition.resid.plot(ax=axes[3], color='#7f8c8d', style='.', legend=False)
axes[3].set_ylabel('Residual')
axes[3].set_xlabel('Tanggal')

plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "seasonal_decomposition.png"), dpi=300)
plt.close()
print("Grafik tren dan dekomposisi waktu berhasil diekspor.")

Grafik tren dan dekomposisi waktu berhasil diekspor.


### Feature Engineering

Membuat fitur lag ($t-1, t-7, t-14, t-30$), rolling average, dan calendar features untuk melatih Random Forest.


In [5]:
df_features = df_clean.copy()

# Fitur lag
df_features['lag_1'] = df_features['Demand_Cleaned'].shift(1)
df_features['lag_7'] = df_features['Demand_Cleaned'].shift(7)
df_features['lag_14'] = df_features['Demand_Cleaned'].shift(14)
df_features['lag_30'] = df_features['Demand_Cleaned'].shift(30)

# Fitur rolling mean
df_features['rolling_mean_7'] = df_features['Demand_Cleaned'].shift(1).rolling(window=7).mean()
df_features['rolling_mean_30'] = df_features['Demand_Cleaned'].shift(1).rolling(window=30).mean()

# Fitur kalender
df_features['day_of_week'] = df_features['Date'].dt.dayofweek
df_features['month'] = df_features['Date'].dt.month
df_features['year'] = df_features['Date'].dt.year
df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)

# Hapus baris NaN
df_features = df_features.dropna().reset_index(drop=True)
print(f"Data siap untuk pemodelan: {df_features.shape}")
df_features.head(2)

Data siap untuk pemodelan: (344, 19)


,Date,Product,Category,Price_IDR,Promotion,Demand_Raw,Stockout_Occurred,Sales_Actual,Demand_Cleaned,lag_1,lag_7,lag_14,lag_30,rolling_mean_7,rolling_mean_30,day_of_week,month,year,is_weekend
0,2024-12-31,SKU_HIJAB_PREMIUM,Fashion,57000,1,0,0,0,0,0.0,0.0,58.0,454.0,0.0,129.400000,1,12,2024,0
1,2025-01-01,SKU_HIJAB_PREMIUM,Fashion,57000,1,0,0,0,0,0.0,0.0,0.0,309.0,0.0,114.266667,2,1,2025,0


### Pemodelan Peramalan Permintaan

Membagi dataset:
- **Training Set**: Tanggal awal hingga 30 September 2025 (sekitar 10 bulan).
- **Testing Set (Holdout)**: 1 Oktober 2025 s/d 14 Desember 2025 (75 hari terakhir).

Model:
1. **Seasonal Naive (Baseline)**: Menggunakan `lag_7` sebagai tebakan.
2. **SARIMAX**: Model statistik linear parametrik dengan harga (`Price_IDR`) dan promosi (`Promotion`) sebagai variabel eksogen.
3. **Random Forest Regressor**: Algoritma machine learning non-linear.


In [6]:
# Split data
split_date = pd.to_datetime("2025-10-01")
train_df = df_features[df_features['Date'] < split_date].copy()
test_df = df_features[df_features['Date'] >= split_date].copy()

# 1. Baseline Model
test_df['Forecast_Baseline'] = test_df['lag_7']

# 2. SARIMAX Model
y_train_sari = train_df['Demand_Cleaned'].values
exog_train_sari = train_df[['Price_IDR', 'Promotion']].values
exog_test_sari = test_df[['Price_IDR', 'Promotion']].values

sarimax_model = SARIMAX(
    y_train_sari,
    exog=exog_train_sari,
    order=(1, 1, 1),
    seasonal_order=(1, 0, 0, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarimax_fit = sarimax_model.fit(disp=False)
test_df['Forecast_SARIMAX'] = sarimax_fit.forecast(steps=len(test_df), exog=exog_test_sari)

# 3. Random Forest Regressor
features = ['lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_mean_30', 
            'Price_IDR', 'Promotion', 'day_of_week', 'month', 'is_weekend']

X_train_rf = train_df[features]
y_train_rf = train_df['Demand_Cleaned']
X_test_rf = test_df[features]

rf_model = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)
rf_model.fit(X_train_rf, y_train_rf)
test_df['Forecast_RF'] = rf_model.predict(X_test_rf)

print("Semua model berhasil dilatih & memprediksi data holdout.")

Semua model berhasil dilatih & memprediksi data holdout.


### Evaluasi Kinerja Peramalan

Menilai akurasi menggunakan MAE, RMSE, dan MAPE.


In [7]:
def calculate_mape(actual, forecast):
    # Hindari pembagian dengan nol
    actual_safe = np.where(actual == 0, 1, actual)
    return np.mean(np.abs((actual - forecast) / actual_safe)) * 100

y_actual = test_df['Demand_Cleaned'].values
models_list = ['Baseline', 'SARIMAX', 'RF']
metrics = {}

for m in models_list:
    col = f'Forecast_{m}'
    pred = test_df[col].values
    mae = mean_absolute_error(y_actual, pred)
    rmse = root_mean_squared_error(y_actual, pred)
    mape = calculate_mape(y_actual, pred)
    metrics[m] = {'MAE': round(mae, 2), 'RMSE': round(rmse, 2), 'MAPE %': round(mape, 2)}

df_metrics = pd.DataFrame(metrics).T
print("TABEL METRIK KINERJA MODEL:")
print(df_metrics)

# Ekspor metrik ke CSV
df_metrics.to_csv(os.path.join(DATA_DIR, "metrics_comparison.csv"), index=True)

# Plot Forecast vs Aktual (Fokus pada 60 hari terakhir agar grafik terbaca jelas)
plot_df = test_df.tail(60).copy()

plt.figure(figsize=(12, 6))
plt.plot(plot_df['Date'], plot_df['Demand_Cleaned'], label='Kebutuhan Aktual (Riil)', color='#2c3e50', linewidth=2, marker='o')
plt.plot(plot_df['Date'], plot_df['Forecast_Baseline'], label='Baseline (Seasonal Naive)', color='#7f8c8d', linestyle='--', alpha=0.6)
plt.plot(plot_df['Date'], plot_df['Forecast_SARIMAX'], label='SARIMAX (Statistik)', color='#d35400', linestyle='-.', alpha=0.8)
plt.plot(plot_df['Date'], plot_df['Forecast_RF'], label='Random Forest (Machine Learning)', color='#27ae60', linewidth=1.5, alpha=0.9)

plt.title('Perbandingan Model Peramalan vs Kebutuhan Aktual SKU_HIJAB_PREMIUM (60 Hari Terakhir)', fontsize=13, fontweight='bold')
plt.xlabel('Tanggal', fontsize=11)
plt.ylabel('Unit Ritel', fontsize=11)
plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "forecast_vs_actual.png"), dpi=300)
plt.close()

# Plot Feature Importance
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
features_sorted = [features[i] for i in indices]

plt.figure(figsize=(10, 5))
sns.barplot(x=importances[indices], y=features_sorted, hue=features_sorted, legend=False, palette='viridis')
plt.title('Fitur Paling Berpengaruh (Feature Importance) - Random Forest', fontsize=13, fontweight='bold')
plt.xlabel('Skor Kepentingan', fontsize=11)
plt.ylabel('Fitur', fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "feature_importance.png"), dpi=300)
plt.close()
print("Grafik model dan metrik evaluasi telah disimpan.")

TABEL METRIK KINERJA MODEL:
             MAE    RMSE   MAPE %
Baseline   98.79  222.89   184.26
SARIMAX   100.61  180.52  1484.46
RF         79.67  161.78   528.83


Grafik model dan metrik evaluasi telah disimpan.


### Optimasi Inventaris (Safety Stock, ROP, & EOQ)

Menerapkan rumus logistik untuk menghitung parameter gudang optimal berdasarkan error model Random Forest.


In [8]:
# 1. Parameter Persediaan Gudang Indonesia
Z = 1.645  # 95% Service Level
L = 5      # Lead Time pengiriman barang domestik (5 hari)

# Hitung standar deviasi kesalahan (residual) dari model peramalan terbaik (Random Forest)
residuals = test_df['Demand_Cleaned'] - test_df['Forecast_RF']
sigma_e = residuals.std()

d = test_df['Demand_Cleaned'].mean() # Rata-rata permintaan harian nasional di periode test
D = d * 365 # Proyeksi permintaan tahunan

# Biaya logistik nasional
S = 750000  # Biaya pengiriman kontainer & penanganan logistik sekali pesan (IDR)
H = 12000   # Biaya simpan per unit per tahun (IDR)

# 2. Hitung Parameter Logistik
safety_stock = int(np.round(Z * sigma_e * np.sqrt(L)))
reorder_point = int(np.round((d * L) + safety_stock))
eoq = int(np.round(np.sqrt((2 * D * S) / H)))

# Simpan metrik logistik ke JSON
inventory_metrics = {
    'Average_Daily_Demand': round(d, 2),
    'Residual_Std_Dev_Units': round(sigma_e, 2),
    'Safety_Stock_Units': safety_stock,
    'Reorder_Point_Units': reorder_point,
    'Economic_Order_Quantity_Units': eoq
}
with open(os.path.join(DATA_DIR, "inventory_metrics.json"), 'w') as f:
    json.dump(inventory_metrics, f, indent=4)

print("HASIL KALKULASI PARAMETER MANAJEMEN INVENTARIS:")
for k, v in inventory_metrics.items():
    print(f"* {k.replace('_', ' ')}: {v}")

# 3. Jalankan Simulasi Tingkat Persediaan Harian (Sawtooth Curve)
current_inv = eoq + safety_stock
order_in_transit = False
days_to_arrival = 0

inventory_levels = []
order_dates = []
arrival_dates = []

dates_sim = test_df['Date'].values
demand_sim = test_df['Demand_Cleaned'].values

for i in range(len(test_df)):
    if order_in_transit:
        days_to_arrival -= 1
        if days_to_arrival == 0:
            current_inv += eoq
            order_in_transit = False
            arrival_dates.append(dates_sim[i])
            
    current_inv = max(0, current_inv - demand_sim[i])
    inventory_levels.append(current_inv)
    
    if current_inv <= reorder_point and not order_in_transit:
        order_in_transit = True
        days_to_arrival = L
        order_dates.append(dates_sim[i])

test_df['Inventory_Level'] = inventory_levels

# Plot Sawtooth Curve
plt.figure(figsize=(12, 6))
plt.plot(test_df['Date'], test_df['Inventory_Level'], label='Tingkat Stok Gudang', color='#1abc9c', linewidth=2)
plt.axhline(y=reorder_point, color='#e67e22', linestyle='--', label=f'Reorder Point (ROP = {reorder_point} unit)', alpha=0.8)
plt.axhline(y=safety_stock, color='#e74c3c', linestyle=':', label=f'Safety Stock (SS = {safety_stock} unit)', alpha=0.8)
plt.fill_between(test_df['Date'], 0, safety_stock, color='#c0392b', alpha=0.1, label='Zona Risiko Stockout')

if order_dates:
    plt.scatter(order_dates, [reorder_point] * len(order_dates), color='#d35400', marker='v', s=80, zorder=5, label='Kirim Rilis Pemesanan Baru (EOQ)')

plt.title('Simulasi Siklus Persediaan Gudang SKU_HIJAB_PREMIUM dengan ROP & EOQ (Semester II 2025)', fontsize=13, fontweight='bold')
plt.xlabel('Tanggal', fontsize=11)
plt.ylabel('Tingkat Stok Inventaris (Unit)', fontsize=11)
plt.legend(loc='upper right', frameon=True)
plt.tight_layout()
plt.savefig(os.path.join(IMAGES_DIR, "safety_stock_rop.png"), dpi=300)
plt.close()
print("Grafik simulasi sawtooth curve persediaan berhasil disimpan.")

HASIL KALKULASI PARAMETER MANAJEMEN INVENTARIS:
* Average Daily Demand: 105.14
* Residual Std Dev Units: 162.94
* Safety Stock Units: 599
* Reorder Point Units: 1125
* Economic Order Quantity Units: 2190


Grafik simulasi sawtooth curve persediaan berhasil disimpan.
